##### Noise Aware Mitigated QSVM - Spambase

In [ ]:
import qiskit, qiskit_aer, qiskit_machine_learning
print("Qiskit:", qiskit.__version__)
print("Aer:", qiskit_aer.__version__)
print("QML:", qiskit_machine_learning.__version__)

In [ ]:
# To ensure reproducibility of results
from qiskit_machine_learning.utils import algorithm_globals
algorithm_globals.random_seed = 12345

In [ ]:
# --- Import Libraries ---
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
# from imblearn.over_sampling import RandomOverSampler  # For optional balancing

In [ ]:
# --- Qiskit Imports ---
from qiskit.circuit.library import ZZFeatureMap
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError
from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC, PegasosQSVC

In [ ]:
# --- Import Spambase Column Names ---
spambase_columns = [
    "word_freq_make",
    "word_freq_address",
    "word_freq_all",
    "word_freq_3d",
    "word_freq_our",
    "word_freq_over",
    "word_freq_remove",
    "word_freq_internet",
    "word_freq_order",
    "word_freq_mail",
    "word_freq_receive",
    "word_freq_will",
    "word_freq_people",
    "word_freq_report",
    "word_freq_addresses",
    "word_freq_free",
    "word_freq_business",
    "word_freq_email",
    "word_freq_you",
    "word_freq_credit",
    "word_freq_your",
    "word_freq_font",
    "word_freq_000",
    "word_freq_money",
    "word_freq_hp",
    "word_freq_hpl",
    "word_freq_george",
    "word_freq_650",
    "word_freq_lab",
    "word_freq_labs",
    "word_freq_telnet",
    "word_freq_857",
    "word_freq_data",
    "word_freq_415",
    "word_freq_85",
    "word_freq_technology",
    "word_freq_1999",
    "word_freq_parts",
    "word_freq_pm",
    "word_freq_direct",
    "word_freq_cs",
    "word_freq_meeting",
    "word_freq_original",
    "word_freq_project",
    "word_freq_re",
    "word_freq_edu",
    "word_freq_table",
    "word_freq_conference",
    "char_freq_;",
    "char_freq_(",
    "char_freq_[",
    "char_freq_!",
    "char_freq_$",
    "char_freq_#",
    "capital_run_length_average",
    "capital_run_length_longest",
    "capital_run_length_total",
    # finally the target label column:
    "label"
]

# --- 1. Load the Spambase Dataset ---
file_path = r'C:\Users\User\Documents\MyProjects\FYP_ResearchProject\data\spambase\spambase.data'
df = pd.read_csv(file_path, header=None, names=spambase_columns)
df.drop_duplicates(inplace=True)

In [ ]:
# 2. Some basic processing
print(f"Original shape of Spambase data: {df.shape}") # Prints original dataset shape
df.drop_duplicates(inplace=True) # Remove duplicates
print(f"Shape after dropping duplicates: {df.shape}\n") # Then print again the new shape

In [ ]:
# Data Preparation

# 1. Split features and target
X = df.drop('label', axis=1)
y = df['label']

# ============================================
# SUBSET DATA (for QSVC - 300 samples)
# ============================================
# First sample 429 samples from full dataset
X_subset, _, y_subset, _ = train_test_split(
    X, y,
    train_size=429,
    stratify=y,
    random_state=42
)

# Then do 70:30 split on this subset
X_train, X_test, y_train, y_test = train_test_split(
    X_subset, y_subset,
    test_size=0.30,
    random_state=42,
    stratify=y_subset
)
# This gives you ~300 training, ~129 test samples for QSVC

print(f"QSVC Training set: {X_train.shape[0]} samples")
print(f"QSVC Test set: {X_test.shape[0]} samples")

# ============================================
# FULL DATA (for PegasosQSVC - 4000+ samples)
# ============================================
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print(f"PegasosQSVC Training set: {X_train_full.shape[0]} samples")
print(f"PegasosQSVC Test set: {X_test_full.shape[0]} samples\n")

In [ ]:
# Scaling for SUBSET (QSVC)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # Use same scaler

X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X.columns)

# Scaling for FULL DATASET (PegasosQSVC)
scaler_full = StandardScaler()
X_train_full_scaled = scaler_full.fit_transform(X_train_full)
X_test_full_scaled = scaler_full.transform(X_test_full)

X_train_full_scaled_df = pd.DataFrame(X_train_full_scaled, columns=X.columns)
X_test_full_scaled_df = pd.DataFrame(X_test_full_scaled, columns=X.columns)

In [ ]:
# --- Feature Correlation Analysis ---
print("--- Feature Correlation Analysis ---")
THRESH = 0.9

# Calculate correlation matrix on the SCALED TRAINING data (subset)
corr_matrix_train = X_train_scaled_df.corr().abs()

# Get the upper triangle of the correlation matrix
upper_triangle = corr_matrix_train.where(np.triu(np.ones(corr_matrix_train.shape), k=1).astype(bool))

# Find features with correlation greater than the threshold
columns_to_drop = set()
for column in upper_triangle.columns:
    high_corr_partners = upper_triangle.index[upper_triangle[column] > THRESH].tolist()
    if high_corr_partners:
        for partner in high_corr_partners:
            # IMPORTANT: Check correlation with the TRAINING target variable
            corr_main_vs_target = y_train.corr(X_train_scaled_df[column])
            corr_partner_vs_target = y_train.corr(X_train_scaled_df[partner])
            
            print(f"Found pair: ('{column}', '{partner}') with correlation > {THRESH}")
            if abs(corr_main_vs_target) < abs(corr_partner_vs_target):
                columns_to_drop.add(column)
                print(f"-> Dropping '{column}' (weaker correlation with target)")
            else:
                columns_to_drop.add(partner)
                print(f"-> Dropping '{partner}' (weaker correlation with target)")

to_drop_final = sorted(list(columns_to_drop))
print(f"\nTotal features to drop ({len(to_drop_final)}): {to_drop_final}")

# Drop the identified columns from SUBSET (QSVC)
X_train_selected = X_train_scaled_df.drop(columns=to_drop_final)
X_test_selected = X_test_scaled_df.drop(columns=to_drop_final)

# Drop the same columns from FULL DATASET (PegasosQSVC)
X_train_full_selected = X_train_full_scaled_df.drop(columns=to_drop_final)
X_test_full_selected = X_test_full_scaled_df.drop(columns=to_drop_final)

print(f"\nOriginal number of features: {X_train.shape[1]}")
print(f"Number of features after selection: {X_train_selected.shape[1]}\n")

In [ ]:
# PCA for SUBSET (QSVC)
n_components = 4
pca = PCA(n_components=n_components, random_state=42)

# Fit on the selected training data and transform both sets
X_train_pca = pca.fit_transform(X_train_selected)
X_test_pca = pca.transform(X_test_selected)

print(f"QSVC - Shape after PCA (Train): {X_train_pca.shape}")
print(f"QSVC - Shape after PCA (Test):  {X_test_pca.shape}")

# PCA for FULL DATASET (PegasosQSVC)
pca_full = PCA(n_components=n_components, random_state=42)
X_train_full_pca = pca_full.fit_transform(X_train_full_selected)
X_test_full_pca = pca_full.transform(X_test_full_selected)

print(f"PegasosQSVC - Shape after PCA (Train): {X_train_full_pca.shape}")
print(f"PegasosQSVC - Shape after PCA (Test):  {X_test_full_pca.shape}")

##### Noise Simulation Setup

In [ ]:
# Noise Model implementation (Depolarizing error and Readout Error)
print("--- Setting up Noise Model ---")
# Error rates (realistic, not too high)
p_gate_1q = 0.001   # 0.1% error for single-qubit gates (u1, u2, u3)
p_gate_2q = 0.01    # 1.0% error for two-qubit gates (cx)
p_readout = 0.02    # 2.0% chance of wrong measurement

def get_scaled_noise_model(scale_factor=1.0):
    """Build a noise model with scaled error probabilities for ZNE."""
    noise_model = NoiseModel()
    
    # Scale probabilities: p_new = 1 - (1-p)^scale
    p_1q_scaled = 1 - (1 - p_gate_1q)**scale_factor
    p_2q_scaled = 1 - (1 - p_gate_2q)**scale_factor
    p_readout_scaled = 1 - (1 - p_readout)**scale_factor
    
    noise_model.add_all_qubit_quantum_error(depolarizing_error(p_1q_scaled, 1), ['u1', 'u2', 'u3'])
    noise_model.add_all_qubit_quantum_error(depolarizing_error(p_2q_scaled, 2), ['cx'])
    
    readout_error = ReadoutError([[1 - p_readout_scaled, p_readout_scaled], [p_readout_scaled, 1 - p_readout_scaled]])
    noise_model.add_all_qubit_readout_error(readout_error)
    
    return noise_model

# Base noise model (scale=1)
noise_model = get_scaled_noise_model(1.0)
print("Base noise model created.")

##### Backends and Samplers Implementation

In [ ]:
# Create Backends and Samplers for ZNE (scale 1.0 and 3.0)
noisy_backend_1 = AerSimulator(
    noise_model=get_scaled_noise_model(1.0),
    seed_simulator=12345,
)
noisy_backend_3 = AerSimulator(
    noise_model=get_scaled_noise_model(3.0),
    seed_simulator=12345,
)

sampler_1 = AerSampler.from_backend(
    backend=noisy_backend_1,
    default_shots=8192,
)
sampler_3 = AerSampler.from_backend(
    backend=noisy_backend_3,
    default_shots=8192,
)

print("Noisy SamplerV2 (scale 1 and 3) ready for ZNE!")

##### Pass Managers Setup

In [ ]:
# Transpilation pass manager
pm_1 = generate_preset_pass_manager(optimization_level=1, backend=noisy_backend_1)
pm_3 = generate_preset_pass_manager(optimization_level=1, backend=noisy_backend_3)

print("Pass managers ready.")

##### Quantum Kernel Implementation

In [ ]:
feature_dim = n_components
fm = ZZFeatureMap(feature_dimension=feature_dim, reps=1, entanglement='linear')

# Fidelity with noisy sampler and transpilation - Scale 1
fidelity_1 = ComputeUncompute(sampler=sampler_1, pass_manager=pm_1)
quantum_kernel_1 = FidelityQuantumKernel(fidelity=fidelity_1, feature_map=fm)

# Fidelity with noisy sampler and transpilation - Scale 3
fidelity_3 = ComputeUncompute(sampler=sampler_3, pass_manager=pm_3)
quantum_kernel_3 = FidelityQuantumKernel(fidelity=fidelity_3, feature_map=fm)

print("Quantum Kernels (scale 1 and 3) created.")

##### Compute Kernel Matrices

In [ ]:
# Compute Kernel Matrices for ZNE
print("Computing kernel matrix at scale 1.0...")
matrix_train_1 = quantum_kernel_1.evaluate(x_vec=X_train_pca)
matrix_test_1 = quantum_kernel_1.evaluate(x_vec=X_test_pca, y_vec=X_train_pca)

print("Computing kernel matrix at scale 3.0...")
matrix_train_3 = quantum_kernel_3.evaluate(x_vec=X_train_pca)
matrix_test_3 = quantum_kernel_3.evaluate(x_vec=X_test_pca, y_vec=X_train_pca)

print("Kernel matrices computed.")

##### Zero Noise Extrapolation (ZNE) Implementation

In [ ]:
# Zero Noise Extrapolation
# Richardson extrapolation with scales [1, 3]:
# K_mitigated = 1.5 * K(scale=1) - 0.5 * K(scale=3)

matrix_train_zne = 1.5 * matrix_train_1 - 0.5 * matrix_train_3
matrix_test_zne = 1.5 * matrix_test_1 - 0.5 * matrix_test_3

print("ZNE extrapolation complete.")

##### Mitigated Kernel Matrix

In [ ]:
# Plotting mitigated kernel matrix
plt.figure(figsize=(8, 6))
plt.imshow(matrix_train_zne, cmap='viridis')
plt.title("Mitigated (ZNE) Kernel Matrix")
plt.colorbar()
plt.show()

##### Mitigated QSVC Implementation

In [ ]:
# Training with precomputed mitigated kernel
print("--- Training Mitigated QSVC (Spambase) ---")
start_time = time.time()

param_grid = {
    'C': [0.1, 1, 10, 100],
}

# Use cross-validation suitable for data
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# Grid search on SVC with precomputed kernel
grid_search = GridSearchCV(
    SVC(kernel='precomputed', class_weight='balanced'),
    param_grid,
    cv=cv,
    scoring='accuracy',
    verbose=1
)

# Get the best C
grid_search.fit(matrix_train_zne, y_train)
qsvc_mitigated = grid_search.best_estimator_
print(f"Best parameters: {grid_search.best_params_}")
end_time = time.time()
print(f"QSVC training finished in {end_time - start_time:.2f} seconds.")

##### Model Evaluation

In [ ]:
y_train_pred = qsvc_mitigated.predict(matrix_train_zne)
train_accuracy = accuracy_score(y_train, y_train_pred)

y_test_pred = qsvc_mitigated.predict(matrix_test_zne)
test_accuracy = accuracy_score(y_test, y_test_pred)

generalization_gap = abs(train_accuracy - test_accuracy)

print(f"\n--- Mitigated (ZNE) QSVM Evaluation (Spambase) ---")
print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Test Accuracy:     {test_accuracy:.4f}")
print(f"Generalization Gap: {generalization_gap:.4f}")
print("\nClassification Report (Test Set):")
print(classification_report(y_test, y_test_pred, zero_division=0))